In [4]:
import os
import glob
import ast
import re
# import matplotlib.pyplot as plt # 移除

# ----------------------------
# 工具函数
# ----------------------------

def read_last_non_empty_line(filepath):
    """读取文件最后一个非空行"""
    try:
        with open(filepath, 'rb') as f:
            f.seek(0, os.SEEK_END)
            file_size = f.tell()
            if file_size == 0:
                return None

            buffer = bytearray()
            pos = file_size - 1
            while pos >= 0:
                f.seek(pos)
                char = f.read(1)
                if char == b'\n':
                    if buffer:
                        line = buffer[::-1].decode('utf-8', errors='ignore')
                        stripped = line.strip()
                        if stripped:
                            return stripped
                        buffer = bytearray()
                else:
                    buffer.append(char[0])
                pos -= 1

            if buffer:
                line = buffer[::-1].decode('utf-8', errors='ignore')
                stripped = line.strip()
                if stripped:
                    return stripped
            return None
    except Exception as e:
        print(f"读取失败 {filepath}: {e}")
        return None

def parse_and_avg_metrics(line):
    """从字典行提取6个指标、返回平均值和原始字典"""
    if not line:
        return None, None
    try:
        data = ast.literal_eval(line)
        metrics = [
            float(data['HIT@5']), float(data['NDCG@5']),
            float(data['HIT@10']), float(data['NDCG@10']),
            float(data['HIT@20']), float(data['NDCG@20'])
        ]
        avg = sum(metrics) / len(metrics)
        return avg, data  # 返回平均值和完整字典
    except (ValueError, SyntaxError, KeyError, TypeError):
        return None, None

def extract_lc_from_filename(filename):
    """
    从文件名如 'Beauty-SASRec-Mamba4Rec_l2.5_k10.log' 中提取 l 值
    返回 float 类型的 l 值，如 2.5
    """
    # 注意：根据您的代码，这里匹配的是 _l 而不是 _lc
    match = re.search(r'_l([0-9]+\.?[0-9]*)_', filename) 
    if match:
        return float(match.group(1))
    else:
        # 尝试匹配末尾无下划线的情况（健壮性）
        match = re.search(r'_l([0-9]+\.?[0-9]*)\.', filename)
        if match:
            return float(match.group(1))
    return None

# ----------------------------
# 主逻辑
# ----------------------------

def collect_lc_and_avg(folder_path, prefix="Beauty-", extension="*.log", recursive=True):
    if recursive:
        pattern = os.path.join(folder_path, "**", f"{prefix}{extension}")
    else:
        pattern = os.path.join(folder_path, f"{prefix}{extension}")
    
    files = glob.glob(pattern, recursive=recursive)
    results = []

    for file in files:
        filename = os.path.basename(file)

        lc = extract_lc_from_filename(filename)
        if lc is None:
            # print(f"⚠️ 无法从文件名提取 l: {filename}")
            continue

        last_line = read_last_non_empty_line(file)
        avg_metric, all_metrics = parse_and_avg_metrics(last_line) # 获取平均值和所有指标
        
        if avg_metric is None:
            # print(f"⚠️ 无法解析指标: {filename}")
            continue

        # 存储所有信息
        results.append((lc, avg_metric, filename, all_metrics))

    # 移除按 lc 排序
    # results.sort(key=lambda x: x[0]) 
    return results

# ----------------------------
# 绘图
# ----------------------------

# 移除了 plot_lc_vs_avg 函数

# ----------------------------
# 主程序
# ----------------------------

if __name__ == "__main__":
    FOLDER = "./TEST_KD"
    PREFIX = "T"
    EXTENSION = "*.log"
    
    results = collect_lc_and_avg(
        folder_path=FOLDER,
        prefix=PREFIX,
        extension=EXTENSION,
        recursive=True
    )

    if not results:
        print("未找到有效数据。")
    else:
        # --- 新增：按平均值从大到小排序 ---
        results.sort(key=lambda x: x[1], reverse=True)

        # --- 修改：打印 Top 5 及所有指标 ---
        print(f"\n📊 Top 5 结果 (按平均指标从大到小排序):")
        
        # 遍历前5名
        for i, (lc, avg, fname, metrics_dict) in enumerate(results[:10]):
            print(f"\n--- 排名 {i + 1} ---")
            print(f"  文件名: {fname}")
            print(f"  平均指标: {avg:.6f} (l={lc})", end=' ')
            print(f"  所有指标:", end=' ')
            # 格式化打印字典
            max_key_len = max(len(k) for k in metrics_dict.keys())
            for key, val in metrics_dict.items():
                # 尝试将浮点数格式化，其他类型直接转换
                try:
                    val_str = f"{float(val):.6f}"
                except (ValueError, TypeError):
                    val_str = str(val)
                print(f"- {key:<{max_key_len}}:{val_str}", end=' ')



📊 Top 5 结果 (按平均指标从大到小排序):

--- 排名 1 ---
  文件名: Toys_and_Games-SASRec-Mamba4Rec_a1.0_b1.0_l3.0_dwsb1.0_t0.0_k1_N100_kdt0.7_kdg0.01.log
  平均指标: 0.096733 (l=3.0)   所有指标: - Epoch  :55.000000 - HIT@5  :0.089600 - NDCG@5 :0.066800 - HIT@10 :0.120000 - NDCG@10:0.076500 - HIT@20 :0.144700 - NDCG@20:0.082800 
--- 排名 2 ---
  文件名: Toys_and_Games-SASRec-Mamba4Rec_a1.0_b1.0_l2.5_dwsb1.0_t0.0_k1_N100_kdt10.0_kdg0.01.log
  平均指标: 0.096450 (l=2.5)   所有指标: - Epoch  :148.000000 - HIT@5  :0.091700 - NDCG@5 :0.064700 - HIT@10 :0.118900 - NDCG@10:0.073500 - HIT@20 :0.148800 - NDCG@20:0.081100 
--- 排名 3 ---
  文件名: Toys_and_Games-SASRec-Mamba4Rec_a1.0_b1.0_l3.0_dwsb1.0_t0.0_k1_N100_kdt10.0_kdg0.01.log
  平均指标: 0.095883 (l=3.0)   所有指标: - Epoch  :146.000000 - HIT@5  :0.088600 - NDCG@5 :0.062800 - HIT@10 :0.116400 - NDCG@10:0.071700 - HIT@20 :0.154500 - NDCG@20:0.081300 
--- 排名 4 ---
  文件名: Toys_and_Games-SASRec-Mamba4Rec_a1.0_b1.0_l1.5_dwsb1.0_t0.0_k1_N100_kdt10.0_kdg0.01.log
  平均指标: 0.095867 (l=1.5)   所有指标: - 

In [2]:
import os
import ast # 用于安全地将字符串字典转换为Python字典
import re

def get_last_non_empty_line(file_path):
    """读取文件并返回最后一行非空内容"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            for line in reversed(lines):
                stripped_line = line.strip()
                if stripped_line:
                    return stripped_line
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
    return None

def parse_metrics_and_calculate_average(line, metrics_to_average):
    """
    从行中提取指标字典并计算指定指标的平均值。
    """
    try:
        # 查找字典开始的 '{'
        dict_start_index = line.find('{')
        if dict_start_index == -1:
            return None, None

        # 提取字典字符串
        dict_str = line[dict_start_index:]
        
        # 使用 ast.literal_eval 安全解析
        metrics_dict = ast.literal_eval(dict_str)

        
        values = []
        for metric in metrics_to_average:
            if metric in metrics_dict:
                try:
                    values.append(float(metrics_dict[metric]))
                except (ValueError, TypeError):
                    print(f"Warning: Could not convert metric '{metric}' value '{metrics_dict[metric]}' to float.")
            else:
                print(f"Warning: Metric '{metric}' not found in {metrics_dict}")

        if not values:
            return None, None
            
        # 计算平均值
        average_score = sum(values) / len(values)
        return average_score, metrics_dict

    except (SyntaxError, ValueError) as e:
        print(f"Error parsing line: {line}\nError: {e}")
        return None, None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None, None

def find_top_5_files(directory, file_extension=".log"):
    """
    遍历目录，解析文件，并找出平均指标最高的5个文件。
    """
    
    # 1. 定义您关心并希望平均化的指标
    # 根据您的示例，我们使用 HIT@K 和 NDCG@K
    metrics_to_average = [
        'HIT@5', 'NDCG@5', 
        'HIT@10', 'NDCG@10', 
        'HIT@20', 'NDCG@20'
    ]
    
    file_scores = []

    # 遍历目录中的所有文件
    for filename in os.listdir(directory):
        if filename.endswith(file_extension):
            file_path = os.path.join(directory, filename)
            
            # 1. 获取最后一行非空行
            last_line = get_last_non_empty_line(file_path)
            
            if not last_line:
                # 如果行不存在或不包含关键字，则跳过
                continue
                
            # 2. 提取指标并计算平均值
            average_score, metrics_dict = parse_metrics_and_calculate_average(last_line, metrics_to_average)
            
            if average_score is not None:
                # 3. 存储结果
                file_scores.append((filename, average_score, metrics_dict))

    # 4. 按平均值降序排序
    file_scores.sort(key=lambda x: x[1], reverse=True)

    # 5. 找出并打印平均值最大的五个文件
    print(f"--- Top 5 Files in '{directory}' (by avg of {metrics_to_average}) ---")
    
    if not file_scores:
        print("No valid metric files found.")
        return

    for i, (filename, avg_score, metrics) in enumerate(file_scores[:5]):
        print(f"\nRank {i+1}: {filename}")
        print(f"  Average Score: {avg_score:.6f}")
        print(f"  Metrics: {metrics}")

# --- 如何使用 ---

# 1. 设置您的日志文件所在的文件夹路径
#    将 'your_logs_directory_path' 替换为您的实际路径
#    例如: r'C:\projects\my_model\logs' 或 '/home/user/experiments/run123'
log_directory = './TEST_KD/' 

# 2. (可选) 设置您的日志文件扩展名，例如 '.txt', '.log' 等
file_ext = '.log' 

# 运行主函数
if log_directory == 'your_logs_directory_path':
    print("="*50)
    
    print("请先在脚本中设置 'log_directory' 变量为您的日志文件夹路径!")
    print("="*50)
else:
    find_top_5_files(log_directory, file_ext)

--- Top 5 Files in './TEST_KD/' (by avg of ['HIT@5', 'NDCG@5', 'HIT@10', 'NDCG@10', 'HIT@20', 'NDCG@20']) ---

Rank 1: LastFM-SASRec-Mamba4Rec_a1.0_b1.0_l2.0_dwsb1.0_t0.0_k1_N100_kdt1.0_kdg0.01.log
  Average Score: 0.187033
  Metrics: {'Epoch': 103, 'HIT@5': '0.1605', 'NDCG@5': '0.1080', 'HIT@10': '0.2388', 'NDCG@10': '0.1332', 'HIT@20': '0.3264', 'NDCG@20': '0.1553'}

Rank 2: LastFM-SASRec-Mamba4Rec_a1.0_b1.0_l0.3_dwsb1.0_t0.0_k1_N100_kdt5.0_kdg0.01.log
  Average Score: 0.160883
  Metrics: {'Epoch': 212, 'HIT@5': '0.1363', 'NDCG@5': '0.0915', 'HIT@10': '0.2033', 'NDCG@10': '0.1131', 'HIT@20': '0.2869', 'NDCG@20': '0.1342'}

Rank 3: LastFM-SASRec-Mamba4Rec_a1.0_b1.0_l0.3_dwsb1.0_t0.0_k1_N100_kdt10.0_kdg0.01.log
  Average Score: 0.158783
  Metrics: {'Epoch': 188, 'HIT@5': '0.1339', 'NDCG@5': '0.0892', 'HIT@10': '0.2019', 'NDCG@10': '0.1113', 'HIT@20': '0.2844', 'NDCG@20': '0.1320'}

Rank 4: LastFM-SASRec-Mamba4Rec_a1.0_b1.0_l0.5_dwsb1.0_t0.0_k1_N100_kdt5.0_kdg0.01.log
  Average Score: 0

In [1]:
import os
import ast # 用于安全地将字符串字典转换为Python字典
import re
import matplotlib
matplotlib.use('Agg') # Use 'Agg' backend for non-interactive plotting
import matplotlib.pyplot as plt

def get_last_non_empty_line(file_path):
    """读取文件并返回最后一行非空内容"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            for line in reversed(lines):
                stripped_line = line.strip()
                if stripped_line:
                    return stripped_line
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
    return None

def parse_metrics_and_calculate_average(line, metrics_to_average):
    """
    从行中提取指标字典并计算指定指标的平均值。
    """
    try:
        # 查找字典开始的 '{'
        dict_start_index = line.find('{')
        if dict_start_index == -1:
            print(f"Warning: No dictionary '{{' found in line: {line}")
            return None, None

        # 提取字典字符串
        dict_str = line[dict_start_index:]
        
        # 使用 ast.literal_eval 安全解析
        metrics_dict = ast.literal_eval(dict_str)

        
        values = []
        for metric in metrics_to_average:
            if metric in metrics_dict:
                try:
                    values.append(float(metrics_dict[metric]))
                except (ValueError, TypeError):
                    print(f"Warning: Could not convert metric '{metric}' value '{metrics_dict[metric]}' to float.")
            else:
                print(f"Warning: Metric '{metric}' not found in {metrics_dict}")

        if not values:
            print("Warning: No valid metrics found to average.")
            return None, None
            
        # 计算平均值
        average_score = sum(values) / len(values)
        return average_score, metrics_dict

    except (SyntaxError, ValueError) as e:
        print(f"Error parsing line: {line}\nError: {e}")
        return None, None
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return None, None

def calculate_and_plot_trends(directory, file_extension, params_to_plot):
    """
    遍历目录，按参数分组计算平均指标，并绘制趋势图。
    """
    
    # 1. 定义您关心并希望平均化的指标
    metrics_to_average = [
        'HIT@5', 'NDCG@5', 
        'HIT@10', 'NDCG@10', 
        'HIT@20', 'NDCG@20'
    ]
    
    # 2. 定义正则表达式来提取参数值
    # 根据 'l0*' 和 [0.1, 0.3, ...]，我们假设文件名为 ..._l00.1_... 等
    # 这个 regex 查找 '_l0' 后面跟着 '0.x' (其中x是任意数字)
    param_regex = re.compile(r'_l(0\.\d)')
    
    # 3. 初始化用于存储每个参数得分的字典
    param_scores = {param: [] for param in params_to_plot}
    print(f"Initializing analysis for params: {params_to_plot}")
    print(f"Looking for files ending in '{file_extension}' in directory '{directory}'")
    print(f"Metrics to average: {metrics_to_average}\n")

    # 遍历目录中的所有文件
    found_files_count = 0
    parsed_files_count = 0
    
    # 确保目录存在
    if not os.path.isdir(directory):
        print(f"Error: Directory not found: {directory}")
        return

    for filename in os.listdir(directory):
        if filename.endswith(file_extension):
            found_files_count += 1
            
            # 4. 尝试从文件名中提取参数
            match = param_regex.search(filename)
            
            if match:
                try:
                    param_str = match.group(1)
                    param_val = float(param_str)
                    
                    # 5. 检查提取的参数是否在我们关心的列表中
                    if param_val in param_scores:
                        file_path = os.path.join(directory, filename)
                        
                        # 6. 获取最后一行
                        last_line = get_last_non_empty_line(file_path)
                        
                        if not last_line:
                            print(f"Skipping {filename}: No content.")
                            continue
                            
                        # 7. 提取指标并计算平均值
                        average_score, _ = parse_metrics_and_calculate_average(last_line, metrics_to_average)
                        
                        if average_score is not None:
                            # 8. 存储结果
                            param_scores[param_val].append(average_score)
                            print(f"  [Param {param_val}] Found {filename}, Score: {average_score:.6f}")
                            parsed_files_count += 1
                        else:
                            print(f"  [Param {param_val}] Found {filename}, but failed to parse metrics from line: {last_line}")
                            
                except ValueError:
                    print(f"Warning: Could not convert extracted param '{param_str}' to float for file {filename}")
                except Exception as e:
                    print(f"Error processing file {filename}: {e}")

    print(f"\n--- Analysis Complete ---")
    print(f"Total files ending in '{file_extension}' found: {found_files_count}")
    print(f"Total files parsed successfully: {parsed_files_count}")

    # 9. 计算每个参数组的最终平均值
    final_results = {}
    print("\n--- Final Group Averages ---")
    for param, scores in param_scores.items():
        if scores:
            avg = sum(scores) / len(scores)
            final_results[param] = avg
            print(f"Param {param}: Average = {avg:.6f} (from {len(scores)} files)")
        else:
            final_results[param] = None # 标记为空，以便绘图时跳过
            print(f"Param {param}: No valid files found or parsed.")

    # 10. 准备绘图数据
    # 过滤掉没有数据的点，并按x轴（参数）排序
    plot_data = sorted([(p, a) for p, a in final_results.items() if a is not None])
    
    if not plot_data:
        print("\nNo data to plot.")
        return

    x_values = [item[0] for item in plot_data]
    y_values = [item[1] for item in plot_data]

    # 11. 绘制趋势图
    try:
        plt.figure(figsize=(10, 6))
        plt.plot(x_values, y_values, marker='o', linestyle='-')
        
        plt.title(f'Average Score Trend by Parameter')
        plt.xlabel('Parameter Value (from _l0* pattern)')
        plt.ylabel(f'Average Score (of {", ".join(metrics_to_average)})')
        
        # 确保x轴显示我们所有关心的原始参数值
        plt.xticks(params_to_plot) 
        
        plt.grid(True, linestyle='--')
        plt.tight_layout()
        
        plot_filename = 'score_trend.png'
        plt.savefig(plot_filename)
        print(f"\nTrend plot saved to {plot_filename}")

    except Exception as e:
        print(f"Error during plotting: {e}")

# --- 如何使用 ---

# 1. 设置您的日志文件所在的文件夹路径
log_directory = 'Others' 

# 2. 设置您的日志文件扩展名
file_ext = '.log' 

# 3. 设置您关心的参数值 (用于x轴)
params_to_plot = [0.1, 0.3, 0.5, 0.7, 0.9, 1.1]

# 运行主函数
calculate_and_plot_trends(log_directory, file_ext, params_to_plot)

Initializing analysis for params: [0.1, 0.3, 0.5, 0.7, 0.9, 1.1]
Looking for files ending in '.log' in directory 'Others'
Metrics to average: ['HIT@5', 'NDCG@5', 'HIT@10', 'NDCG@10', 'HIT@20', 'NDCG@20']


--- Analysis Complete ---
Total files ending in '.log' found: 66
Total files parsed successfully: 0

--- Final Group Averages ---
Param 0.1: No valid files found or parsed.
Param 0.3: No valid files found or parsed.
Param 0.5: No valid files found or parsed.
Param 0.7: No valid files found or parsed.
Param 0.9: No valid files found or parsed.
Param 1.1: No valid files found or parsed.

No data to plot.
